# Making sure reduction functions are never repeated in Cherry Tables by changing the searchspace

## Imports

In [47]:
from hashlib import sha256
import mmh3
import time
from tqdm import tqdm
import numpy as np
import random
from math import log, e
import pickle

## Helper Functions

In [48]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        print("No startpoints file found - generating new startpoints.")
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

In [49]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_max = alpha * ((2*N)/t+2) # our target mt
m_0 = round(mt_max/(1-alpha))    # m_0 - number of startpoints

In [50]:
# get # cherry-picks per column from pickle file 

# load pickle file
with open(f'higher_costs_ftol_1.pickle', 'rb') as f:
    data = pickle.load(f)

# structure of file
# array of different alpha used
    # for each alpha, array of different costs used
    # for each cost, array arranged as [Kjs, m_0, final_cost, m_values]

# to get Kjs for alpha = 0.95 and cost factor = 5
Kis = data[-1][0][0]

In [51]:
def H(x):
    return sha256(x.to_bytes(8, 'little')).digest()  # return bytes directly

def r(N, t, y, i, ell=0):
    seed = int(i) + (ell*t)
    return mmh3.hash(y, seed, signed=False) % N

## Adapted Cherry Table Function

- Make the searchspace 16 bit integer
- Pick a random point in searchspace to trial
- Check it hasn't been used before in this trial
    - random.sample(range(2^16), k = k_i, replace=false)
    - then iterate through this to get all the reduction function trials
- merge the column number to it to make 32 bit integer, and make sure all the rf indexes are unique

In [52]:
def build_cherry_table(N, t, startpoints, Kis):

    # parameters to store how many hashes and reductions are done, as well as actual time it takes to make this table
    hashes = 0
    reductions = 0
    duration = 0

    # get parameters for reduction function search
    t_bits = (t-1).bit_length() # number of bits to represent t
    k_bits = 32 - t_bits  # number of bits to represent rf index
    max_k_index = 2**k_bits  # maximum number of reduction functions per column

    # start monitoring time
    start = time.perf_counter()


    # store table in dictionary
    # initialise the table with sp:sp pairs
    table = {sp: sp for sp in startpoints}  # store all the points then remove duplicate entries - can't do duplicate keys in dictionary anyway so we can just store all ep:sp

    # Instead of making the table chain by chain, we have to make it column by column to test what reduction function to choose
    # store reduction function indexes
    rf_indexes = []

    # for each column
    for i in tqdm(range(t), desc=f"Calculating columns: "):

        # variable to store best cherry-pick
        best_trial = -1

        # hash all current points then store with startpoints - this stores all our current points
        hashed_points = {H(mi): sp for mi, sp in table.items()}
        hashes += (len(startpoints))    # increment hashes count

        # we are going to continuously replace table with the best rf trial, so we empty it for now
            # we haven't lost the current points as we have them hashed in the hashed_points dictionary
        table = {}

        # get # cherry-picks for this column
        k_i = round(Kis[i])

        # get indexes to trial
        pick_sample = np.array(random.sample(range(max_k_index), k_i), dtype=np.uint32)
        index_sample = (i << k_bits) | pick_sample

        # trial all the reduction functions for this column
        for rf_trial in index_sample:

            # create a trial column to store results of current trial
            trial_column = {}

            # go through each key in hashed_points and store its reduction with sp
            for x in hashed_points:
                # reduce the hash and store in column
                trial_column[r(N, t, x, rf_trial)] = hashed_points[x]
                reductions += 1

            # if trial_column is bigger than current table stored, we replace it
            if len(trial_column) > len(table):
                # replace it 
                table = trial_column
                # replace best cherry-pick
                best_trial = rf_trial

        # now store the best rf cherry pick
        rf_indexes.append(best_trial)


    # finished making table so stop recording time
    duration = time.perf_counter() - start

    # finished, so return table and rf indexes
    return table, rf_indexes, hashes, reductions, duration

## Run

In [53]:
table, rf_indexes, hashes, reductions, duration = build_cherry_table(N, t, get_startpoints(N, m_0, nlabel, alpha), Kis)

No startpoints file found - generating new startpoints.


Calculating columns: 100%|██████████| 80/80 [05:46<00:00,  4.33s/it]
